In [3]:
"""
计算六个LST情景下17个自变量的VIF（方差膨胀因子）。

重要说明
--------
1. VIF只针对自变量计算，因变量本身不计算VIF；
2. 每个LST情景分别确定有效样本：
   - 当前因变量中的0视为缺失；
   - 因变量或任一自变量缺失时删除整行；
   - 不进行任何缺失值填补；
3. 如果六个情景最终使用完全相同的样本，VIF结果也会完全相同；
4. 一般可参考：
   VIF < 5       ：通常不存在严重多重共线性；
   5 <= VIF < 10：存在中等或值得关注的共线性；
   VIF >= 10     ：通常认为存在严重多重共线性。
5. 阈值只是经验标准，应结合变量相关性、理论意义及后续模型稳定性判断。

首次运行若缺少依赖，可在Jupyter Notebook中执行：
!pip install pandas numpy statsmodels openpyxl xlrd matplotlib
"""

# =============================================================================
# 0. 导入库
# =============================================================================
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")


# =============================================================================
# 1. 文件路径
# =============================================================================
DATA_PATH = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\data"
    r"\wuhuanshiliang\SH_attributes.xls"
)

OUTPUT_DIR = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\脚本\Python\vif"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 2. 因变量与自变量
# =============================================================================
TARGETS = [
    "lst0603",
    "lst0806",
    "lst0828",
    "lst0803",
    "lst0619",
    "lst0719",
]

INDEPENDENT_VARS = [
    "TCC",
    "GCI",
    "BCI",
    "WCI",
    "RCI",
    "Shape_Area",
    "jungong",
    "BAH",
    "BHSD",
    "DIST",
    "TCC_b90m",
    "GCI_b90m",
    "BCI_b90m",
    "WCI_b90m",
    "RCI_b90m",
    "BAH_b90m",
    "BHSD_b90m",
]


# =============================================================================
# 3. 工具函数
# =============================================================================
def validate_columns(df: pd.DataFrame) -> None:
    """检查计算所需字段是否全部存在。"""
    required = set(TARGETS + INDEPENDENT_VARS)
    missing = sorted(required.difference(df.columns))

    if missing:
        raise KeyError(
            "数据中缺少以下必要字段：\n"
            + "\n".join(missing)
        )


def classify_vif(vif: float) -> str:
    """根据常用经验阈值标记VIF等级。"""
    if not np.isfinite(vif):
        return "Infinite / perfect collinearity"
    if vif < 5:
        return "Low"
    if vif < 10:
        return "Moderate"
    return "Severe"


def prepare_complete_cases(
    df: pd.DataFrame,
    target: str,
) -> tuple[pd.DataFrame, dict]:
    """
    为当前因变量准备完整案例样本。

    当前因变量中的0转换为NaN；
    随后删除因变量或任一自变量缺失的整行。
    """
    selected_columns = [target] + INDEPENDENT_VARS

    model_df = df[selected_columns].copy()
    model_df = model_df.apply(
        pd.to_numeric,
        errors="coerce",
    )
    model_df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True,
    )

    n_original = len(model_df)

    # 当前因变量中的0代表缺失
    lst_zero_mask = model_df[target].eq(0)
    n_lst_zero = int(lst_zero_mask.sum())
    model_df.loc[lst_zero_mask, target] = np.nan

    complete_mask = model_df.notna().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_removed = int(n_original - n_complete)

    complete_df = model_df.loc[
        complete_mask
    ].copy()

    cleaning_summary = {
        "LST": target,
        "original_samples": n_original,
        "lst_zero_removed": n_lst_zero,
        "total_rows_removed": n_removed,
        "complete_cases": n_complete,
    }

    if n_complete <= len(INDEPENDENT_VARS) + 1:
        raise ValueError(
            f"{target}完整案例数仅为{n_complete}，"
            "不足以稳定计算17个自变量的VIF。"
        )

    return complete_df, cleaning_summary


def calculate_vif(
    complete_df: pd.DataFrame,
    target: str,
) -> pd.DataFrame:
    """
    计算当前情景下所有自变量的VIF与Tolerance。

    VIF_j = 1 / (1 - R_j²)
    Tolerance_j = 1 / VIF_j
    """
    X = complete_df[INDEPENDENT_VARS].copy()

    # 检查零方差变量
    zero_variance_vars = [
        variable
        for variable in INDEPENDENT_VARS
        if np.isclose(
            X[variable].std(ddof=0),
            0,
        )
    ]

    if zero_variance_vars:
        raise ValueError(
            f"{target}存在零方差自变量："
            + ", ".join(zero_variance_vars)
        )

    # 标准化不是计算VIF的必要条件，但可改善不同量纲下的数值稳定性。
    # 标准化不会改变VIF。
    X_std = (
        X - X.mean(axis=0)
    ) / X.std(axis=0, ddof=0)

    X_array = X_std.to_numpy(dtype=float)

    vif_values = []
    for column_index, variable in enumerate(
        INDEPENDENT_VARS
    ):
        vif = variance_inflation_factor(
            X_array,
            column_index,
        )

        tolerance = (
            1.0 / vif
            if np.isfinite(vif) and vif != 0
            else 0.0
        )

        vif_values.append({
            "LST": target,
            "Variable": variable,
            "VIF": float(vif),
            "Tolerance": float(tolerance),
            "VIF_Level": classify_vif(vif),
            "n_samples": int(len(X)),
        })

    vif_df = pd.DataFrame(vif_values)

    # 按VIF从高到低排序
    vif_df = vif_df.sort_values(
        "VIF",
        ascending=False,
    ).reset_index(drop=True)

    return vif_df


def make_model_summary(
    vif_df: pd.DataFrame,
    cleaning_summary: dict,
) -> dict:
    """汇总每个LST情景的共线性诊断。"""
    max_row = vif_df.iloc[0]

    n_vif_ge_5 = int(
        (vif_df["VIF"] >= 5).sum()
    )
    n_vif_ge_10 = int(
        (vif_df["VIF"] >= 10).sum()
    )

    if n_vif_ge_10 > 0:
        conclusion = (
            "At least one predictor has VIF >= 10; "
            "severe multicollinearity may be present."
        )
    elif n_vif_ge_5 > 0:
        conclusion = (
            "No predictor has VIF >= 10, but at least one "
            "predictor has VIF >= 5; multicollinearity "
            "requires attention."
        )
    else:
        conclusion = (
            "All predictors have VIF < 5; no severe "
            "multicollinearity is indicated by the usual criterion."
        )

    return {
        **cleaning_summary,
        "max_VIF": float(max_row["VIF"]),
        "variable_with_max_VIF": str(
            max_row["Variable"]
        ),
        "mean_VIF": float(vif_df["VIF"].mean()),
        "median_VIF": float(
            vif_df["VIF"].median()
        ),
        "number_VIF_ge_5": n_vif_ge_5,
        "number_VIF_ge_10": n_vif_ge_10,
        "conclusion": conclusion,
    }


def plot_vif(
    vif_df: pd.DataFrame,
    target: str,
) -> None:
    """绘制当前情景的VIF水平柱状图。"""
    plot_df = vif_df.sort_values(
        "VIF",
        ascending=True,
    ).copy()

    fig, ax = plt.subplots(
        figsize=(9.5, 7.2),
        dpi=180,
    )

    bars = ax.barh(
        plot_df["Variable"],
        plot_df["VIF"],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.85,
    )

    max_vif = plot_df["VIF"].replace(
        [np.inf, -np.inf],
        np.nan,
    ).max()

    if not np.isfinite(max_vif):
        max_vif = 10

    for bar, value in zip(
        bars,
        plot_df["VIF"],
    ):
        if np.isfinite(value):
            label = f"{value:.2f}"
            x_position = value + max_vif * 0.015
        else:
            label = "Inf"
            x_position = max_vif * 1.01

        ax.text(
            x_position,
            bar.get_y() + bar.get_height() / 2,
            label,
            va="center",
            ha="left",
            fontsize=10,
            fontweight="bold",
        )

    # 常用参考线
    ax.axvline(
        5,
        linestyle="--",
        linewidth=1.2,
        label="VIF = 5",
    )
    ax.axvline(
        10,
        linestyle=":",
        linewidth=1.2,
        label="VIF = 10",
    )

    ax.set_xlabel(
        "Variance Inflation Factor (VIF)",
        fontsize=12,
        fontweight="bold",
    )
    ax.set_ylabel("")
    ax.set_title(
        f"Multicollinearity diagnosis: {target}",
        fontsize=14,
        fontweight="bold",
    )

    ax.tick_params(
        axis="both",
        labelsize=10.5,
        width=1.1,
        direction="out",
    )

    for label in (
        ax.get_xticklabels()
        + ax.get_yticklabels()
    ):
        label.set_fontweight("bold")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.1)
    ax.spines["bottom"].set_linewidth(1.1)
    ax.grid(False)
    ax.legend(
        frameon=False,
        loc="lower right",
    )

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR / f"VIF_{target}.png",
        dpi=600,
        bbox_inches="tight",
    )
    fig.savefig(
        OUTPUT_DIR / f"VIF_{target}.pdf",
        bbox_inches="tight",
    )

    plt.close(fig)


# =============================================================================
# 4. 主程序
# =============================================================================
def main() -> None:
    print("=" * 80)
    print("读取数据")
    print("=" * 80)
    print(f"数据文件：{DATA_PATH}")

    df = pd.read_excel(
        DATA_PATH,
        sheet_name=0,
        engine="xlrd",
    )

    print(
        f"数据维度：{df.shape[0]}行 × "
        f"{df.shape[1]}列"
    )

    validate_columns(df)
    print("字段检查通过。")

    all_vif_results = []
    all_model_summaries = []

    for target in TARGETS:
        print("\n" + "=" * 80)
        print(f"Processing: {target}")
        print("=" * 80)

        complete_df, cleaning_summary = (
            prepare_complete_cases(
                df=df,
                target=target,
            )
        )

        print(
            f"原始样本数：{cleaning_summary['original_samples']}"
        )
        print(
            f"LST=0数量：{cleaning_summary['lst_zero_removed']}"
        )
        print(
            f"删除总行数：{cleaning_summary['total_rows_removed']}"
        )
        print(
            f"完整案例数：{cleaning_summary['complete_cases']}"
        )

        vif_df = calculate_vif(
            complete_df=complete_df,
            target=target,
        )

        model_summary = make_model_summary(
            vif_df=vif_df,
            cleaning_summary=cleaning_summary,
        )

        all_vif_results.append(vif_df)
        all_model_summaries.append(
            model_summary
        )

        print("\nVIF结果：")
        print(
            vif_df[
                [
                    "Variable",
                    "VIF",
                    "Tolerance",
                    "VIF_Level",
                ]
            ].to_string(
                index=False,
                float_format=lambda x: f"{x:.4f}",
            )
        )

        print("\n情景结论：")
        print(model_summary["conclusion"])

        plot_vif(
            vif_df=vif_df,
            target=target,
        )

    # 汇总
    vif_all_df = pd.concat(
        all_vif_results,
        ignore_index=True,
    )

    summary_df = pd.DataFrame(
        all_model_summaries
    )

    # 保存CSV
    vif_all_df.to_csv(
        OUTPUT_DIR / "VIF_all_LST_models.csv",
        index=False,
        encoding="utf-8-sig",
    )

    summary_df.to_csv(
        OUTPUT_DIR / "VIF_model_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # 保存Excel
    excel_path = OUTPUT_DIR / "VIF_all_results.xlsx"

    with pd.ExcelWriter(
        excel_path,
        engine="openpyxl",
    ) as writer:
        vif_all_df.to_excel(
            writer,
            sheet_name="all_VIF",
            index=False,
        )

        summary_df.to_excel(
            writer,
            sheet_name="model_summary",
            index=False,
        )

        # 每个LST单独一个sheet
        for target in TARGETS:
            target_df = vif_all_df.loc[
                vif_all_df["LST"] == target
            ].copy()

            target_df.to_excel(
                writer,
                sheet_name=target[:31],
                index=False,
            )

    print("\n" + "=" * 80)
    print("全部VIF计算完成")
    print("=" * 80)
    print(f"结果目录：{OUTPUT_DIR}")
    print(f"汇总Excel：{excel_path}")

    print("\n各情景最大VIF：")
    print(
        summary_df[
            [
                "LST",
                "variable_with_max_VIF",
                "max_VIF",
                "number_VIF_ge_5",
                "number_VIF_ge_10",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )


if __name__ == "__main__":
    main()

读取数据
数据文件：E:\excel\FILES\博士申请\第二篇论文\data\wuhuanshiliang\SH_attributes.xls
数据维度：1337行 × 164列
字段检查通过。

Processing: lst0603
原始样本数：1337
LST=0数量：0
删除总行数：0
完整案例数：1337

VIF结果：
  Variable    VIF  Tolerance VIF_Level
  BAH_b90m 3.8700     0.2584       Low
 BHSD_b90m 3.5640     0.2806       Low
  BCI_b90m 3.3421     0.2992       Low
      BHSD 2.5965     0.3851       Low
       BAH 2.5543     0.3915       Low
  RCI_b90m 2.1352     0.4683       Low
  TCC_b90m 1.9779     0.5056       Low
  GCI_b90m 1.6843     0.5937       Low
       BCI 1.6268     0.6147       Low
       TCC 1.4802     0.6756       Low
      DIST 1.3654     0.7324       Low
Shape_Area 1.3606     0.7350       Low
       GCI 1.3379     0.7475       Low
   jungong 1.2288     0.8138       Low
  WCI_b90m 1.1909     0.8397       Low
       RCI 1.1317     0.8837       Low
       WCI 1.0614     0.9422       Low

情景结论：
All predictors have VIF < 5; no severe multicollinearity is indicated by the usual criterion.

Processing: lst0806
原始样本数：1

In [2]:
# =============================================================================
# 检查六个LST字段中是否存在0值（缺失值）
# =============================================================================

from pathlib import Path
import pandas as pd

# -----------------------------------------------------------------------------
# 1. 文件路径
# -----------------------------------------------------------------------------
data_path = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\data"
    r"\wuhuanshiliang\SH_attributes.xls"
)

output_dir = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\脚本\Python"
)
output_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. 六个LST字段
# -----------------------------------------------------------------------------
lst_vars = [
    "lst0603",
    "lst0806",
    "lst0828",
    "lst0803",
    "lst0619",
    "lst0719",
]

# -----------------------------------------------------------------------------
# 3. 读取数据
# -----------------------------------------------------------------------------
df = pd.read_excel(
    data_path,
    sheet_name=0,
    engine="xlrd",
)

print(f"数据维度：{df.shape[0]} 行 × {df.shape[1]} 列")

# -----------------------------------------------------------------------------
# 4. 检查字段
# -----------------------------------------------------------------------------
missing_cols = [
    col for col in lst_vars
    if col not in df.columns
]

if missing_cols:
    raise KeyError(
        "缺少字段："
        + ", ".join(missing_cols)
    )

# -----------------------------------------------------------------------------
# 5. 转为数值
# -----------------------------------------------------------------------------
lst_df = df[lst_vars].apply(
    pd.to_numeric,
    errors="coerce",
)

# -----------------------------------------------------------------------------
# 6. 每个日期统计
# -----------------------------------------------------------------------------
zero_summary = pd.DataFrame({
    "LST": lst_vars,
    "Zero_Count": [
        int((lst_df[col] == 0).sum())
        for col in lst_vars
    ],
    "Zero_Percent": [
        float((lst_df[col] == 0).mean() * 100)
        for col in lst_vars
    ],
    "NaN_Count": [
        int(lst_df[col].isna().sum())
        for col in lst_vars
    ],
})

zero_summary["Has_Zero"] = (
    zero_summary["Zero_Count"] > 0
)

print("\n==============================")
print("各日期LST检查")
print("==============================")

print(
    zero_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}",
    )
)

# -----------------------------------------------------------------------------
# 7. 找出存在0值的社区
# -----------------------------------------------------------------------------
zero_mask = lst_df.eq(0)

rows_with_zero = zero_mask.any(axis=1)

n_rows_with_zero = int(rows_with_zero.sum())

print("\n存在至少一个LST=0的社区数量：", n_rows_with_zero)

display_cols = [
    "Excel_Row",
    *lst_vars,
    "Zero_LST_Count",
    "Zero_LST_Dates",
]

if n_rows_with_zero > 0:

    zero_records = df.loc[
        rows_with_zero
    ].copy()

    zero_records.insert(
        0,
        "Excel_Row",
        zero_records.index + 2,
    )

    selected_zero_mask = zero_mask.loc[
        rows_with_zero
    ]

    zero_records["Zero_LST_Count"] = (
        selected_zero_mask.sum(axis=1)
        .to_numpy()
    )

    zero_date_strings = []

    for _, row in selected_zero_mask.iterrows():

        zero_dates = [
            col
            for col in lst_vars
            if row[col]
        ]

        zero_date_strings.append(
            ", ".join(zero_dates)
        )

    zero_records["Zero_LST_Dates"] = zero_date_strings

    print("\nLST=0社区明细：")

    print(
        zero_records[
            display_cols
        ].to_string(index=False)
    )

else:

    print("\n✓ 六个日期中没有任何社区LST等于0。")

    zero_records = pd.DataFrame(
        columns=display_cols
    )

# -----------------------------------------------------------------------------
# 8. 是否多个日期同时为0
# -----------------------------------------------------------------------------
multiple_zero_mask = (
    zero_mask.sum(axis=1) >= 2
)

print(
    "至少两个日期LST=0的社区数量：",
    int(multiple_zero_mask.sum())
)

# -----------------------------------------------------------------------------
# 9. 保存结果
# -----------------------------------------------------------------------------
csv_summary = (
    output_dir /
    "LST_zero_value_summary.csv"
)

csv_records = (
    output_dir /
    "LST_zero_value_records.csv"
)

excel_path = (
    output_dir /
    "LST_zero_value_check.xlsx"
)

zero_summary.to_csv(
    csv_summary,
    index=False,
    encoding="utf-8-sig",
)

zero_records.to_csv(
    csv_records,
    index=False,
    encoding="utf-8-sig",
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
) as writer:

    zero_summary.to_excel(
        writer,
        sheet_name="Zero_Summary",
        index=False,
    )

    zero_records.to_excel(
        writer,
        sheet_name="Zero_Records",
        index=False,
    )

print("\n==============================")
print("检查完成")
print("==============================")

print("统计结果：")
print(csv_summary)

print("社区明细：")
print(csv_records)

print("Excel：")
print(excel_path)

数据维度：1337 行 × 164 列

各日期LST检查
    LST  Zero_Count  Zero_Percent  NaN_Count  Has_Zero
lst0603           0          0.00          0     False
lst0806           0          0.00          0     False
lst0828           0          0.00          0     False
lst0803           0          0.00          0     False
lst0619           0          0.00          0     False
lst0719           0          0.00          0     False

存在至少一个LST=0的社区数量： 0

✓ 六个日期中没有任何社区LST等于0。
至少两个日期LST=0的社区数量： 0

检查完成
统计结果：
E:\excel\FILES\博士申请\第二篇论文\脚本\Python\LST_zero_value_summary.csv
社区明细：
E:\excel\FILES\博士申请\第二篇论文\脚本\Python\LST_zero_value_records.csv
Excel：
E:\excel\FILES\博士申请\第二篇论文\脚本\Python\LST_zero_value_check.xlsx
